# Полносвязный перцептрон с нуля

Нейронная сеть - это функция с параметрами. Она принимает вектор чисел на входе и возвращает число, вектор чисел или класс на выходе. В отличие от обычной заранее выписанной формулы, нейронная сеть обычно содержит большое количество параметров, которые можно подбирать по данным. Именно поэтому одну и ту же архитектуру можно применять к разным задачам: меняются не правила вычисления, а конкретные значения весов и смещений.

В этом ноутбуке строится маленькая полносвязная нейронная сеть без ML-библиотек. Весь прямой проход написан вручную: через списки, циклы, функции и обычную арифметику. Такой способ полезен в начале изучения, потому что он убирает ощущение "черного ящика": сеть оказывается не магическим объектом, а последовательностью понятных вычислений.

Backpropagation здесь намеренно не используется. Сначала важно разобрать, **что именно вычисляет сеть**, из каких математических операций она состоит, почему слои можно рассматривать как композицию функций и откуда появляется способность приближать сложные зависимости.

## 1. Нейронная сеть как функция

Пусть объект описан числовым вектором:

```text
x = (x1, x2, ..., xn)
```

Нейронная сеть задает отображение:

```text
F: R^n -> R^m
```

То есть сеть берет `n` входных чисел и возвращает `m` выходных чисел. В этом смысле нейронная сеть является обычной математической функцией. Особенность состоит в том, что эта функция имеет специальную структуру: она составлена из слоев, а каждый слой выполняет похожую операцию над вектором.

Например, для бинарной классификации часто используют:

```text
F: R^n -> [0, 1]
```

Тогда выход можно интерпретировать как степень уверенности модели в классе `1`. Если выход близок к `0`, модель склоняется к первому варианту; если близок к `1`, ко второму. При этом сама сеть не "понимает" смысл классов в человеческом смысле. Она вычисляет число по заданной формуле.

Слово **параметрическая** означает, что внутри функции есть настраиваемые числа:

- **веса** `w` - коэффициенты при входных признаках;
- **смещения** `b` - свободные члены, которые сдвигают вычисление;
- иногда также говорят просто: параметры сети `theta`.

Если записать это компактно:

```text
prediction = F(x; theta)
```

Здесь `x` - данные, `theta` - все веса и смещения, `prediction` - ответ сети.

Важно отделять переменные от параметров. Вход `x` меняется от объекта к объекту: один день облачный, другой ясный; один пациент имеет одни показатели, другой - другие. Параметры `theta` фиксированы во время прямого прохода и задают поведение модели. Обучение, которое будет рассматриваться отдельно, как раз и состоит в подборе этих параметров.

В этом ноутбуке параметры будут заданы вручную. Это позволяет сосредоточиться на устройстве функции `F`, не смешивая его с вопросом о том, как именно параметры находятся.

## 2. Мини-датасет

Создадим игрушечную задачу: нужно решить, **брать ли зонт**.

У каждого дня есть три признака:

- `cloudiness` - облачность от `0.0` до `1.0`;
- `humidity` - влажность от `0.0` до `1.0`;
- `wind` - ветер от `0.0` до `1.0`.

Целевая переменная:

- `umbrella = 1` - зонт нужен;
- `umbrella = 0` - зонт не нужен.

Это не настоящая метеорологическая модель, а небольшой контролируемый пример. Он нужен для того, чтобы видеть все вычисления и не прятать идею сети за большим объемом данных. На реальных задачах признаков могут быть сотни или тысячи, но принцип остается тем же: объект превращается в числовой вектор, а модель вычисляет по нему ответ.

Обратите внимание, что все признаки уже приведены к диапазону от `0` до `1`. Это удобно, потому что веса разных признаков становятся проще сравнивать. Если один признак измеряется в долях, другой в тысячах, а третий в миллионах, то один только масштаб может начать доминировать в вычислениях. Поэтому нормализация и стандартизация признаков - важная часть подготовки данных.

In [ ]:
dataset = [
    {"day": "A", "cloudiness": 0.10, "humidity": 0.20, "wind": 0.10, "umbrella": 0},
    {"day": "B", "cloudiness": 0.20, "humidity": 0.30, "wind": 0.80, "umbrella": 0},
    {"day": "C", "cloudiness": 0.40, "humidity": 0.50, "wind": 0.20, "umbrella": 0},
    {"day": "D", "cloudiness": 0.65, "humidity": 0.70, "wind": 0.20, "umbrella": 1},
    {"day": "E", "cloudiness": 0.80, "humidity": 0.60, "wind": 0.30, "umbrella": 1},
    {"day": "F", "cloudiness": 0.90, "humidity": 0.90, "wind": 0.70, "umbrella": 1},
    {"day": "G", "cloudiness": 0.30, "humidity": 0.85, "wind": 0.90, "umbrella": 1},
    {"day": "H", "cloudiness": 0.55, "humidity": 0.40, "wind": 0.10, "umbrella": 0},
]

def print_dataset(rows):
    header = f"{'day':<4} {'cloud':>7} {'humid':>7} {'wind':>7} {'target':>7}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['day']:<4} "
            f"{row['cloudiness']:>7.2f} "
            f"{row['humidity']:>7.2f} "
            f"{row['wind']:>7.2f} "
            f"{row['umbrella']:>7}"
        )

print_dataset(dataset)

### Признаки и вектор

Нейронная сеть обычно не работает со словарями напрямую. Ей удобнее получить **вектор чисел**.

Для одного дня:

```text
x = [cloudiness, humidity, wind]
```

Например, день `D`:

```text
x = [0.65, 0.70, 0.20]
```

Векторизация - один из центральных приемов машинного обучения. Реальный объект сначала описывается набором измеримых характеристик, затем эти характеристики упорядочиваются в вектор, а уже после этого модель работает с ним как с точкой в пространстве признаков.

Если признаков три, каждый объект можно представить точкой в трехмерном пространстве. Если признаков сто, объект становится точкой в стомерном пространстве. Человеку трудно представить такое пространство геометрически, но формулы со скалярными произведениями и матрицами работают одинаково при любом числе измерений.

In [ ]:
def row_to_vector(row):
    return [row["cloudiness"], row["humidity"], row["wind"]]

example = dataset[3]
x = row_to_vector(example)

print("День:", example["day"])
print("Вектор признаков:", x)
print("Правильный ответ:", example["umbrella"])

## 3. Один искусственный нейрон

Пусть входной вектор состоит из трех признаков:

```text
x = (x1, x2, x3)
```

У нейрона есть три веса и одно смещение:

```text
w = (w1, w2, w3),    b = const
```

Сначала нейрон считает **аффинное преобразование**:

```text
z = w1*x1 + w2*x2 + w3*x3 + b
```

То же самое через скалярное произведение:

```text
z = w · x + b
```

После этого применяется функция активации `sigma`:

```text
a = sigma(z) = sigma(w · x + b)
```

Где:

- `x` - входной вектор;
- `w` - вектор весов;
- `b` - смещение;
- `z` - значение до активации;
- `a` - выход нейрона после активации.

С математической точки зрения нейрон состоит из двух частей. Первая часть линейно комбинирует признаки: каждый признак получает свой коэффициент, затем результаты складываются. Вторая часть применяет нелинейное преобразование. Эта простая конструкция оказывается очень мощной, когда нейронов много и они соединены в слои.

Без активации нейрон был бы просто линейной формулой с дополнительным свободным членом. Активация позволяет строить нелинейные зависимости.

In [ ]:
import math

def weighted_sum(inputs, weights, bias):
    total = bias
    for x_i, w_i in zip(inputs, weights):
        total += x_i * w_i
    return total

def step(z):
    return 1 if z >= 0 else 0

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

def relu(z):
    return max(0, z)

inputs = [0.65, 0.70, 0.20]
weights = [3.0, 2.0, 0.5]
bias = -2.5

z = weighted_sum(inputs, weights, bias)

print("inputs =", inputs)
print("weights =", weights)
print("bias =", bias)
print("z =", round(z, 4))
print("step(z) =", step(z))
print("sigmoid(z) =", round(sigmoid(z), 4))
print("relu(z) =", round(relu(z), 4))

### Геометрический смысл весов и смещения

Если использовать ступенчатую активацию:

```text
step(z) = 1, если z >= 0
step(z) = 0, если z < 0
```

то нейрон делит пространство входов на две части:

```text
w · x + b = 0
```

Это уравнение задает границу решения.

Для двух признаков граница будет прямой:

```text
w1*x1 + w2*x2 + b = 0
```

Для трех признаков - плоскостью:

```text
w1*x1 + w2*x2 + w3*x3 + b = 0
```

В более высоких размерностях такую границу называют гиперплоскостью.

Вектор весов `w` задает ориентацию этой границы. Если изменить соотношение весов, граница повернется. Смещение `b` сдвигает границу, не меняя ее направления. Поэтому веса отвечают не только за "важность признаков", но и за геометрию разделения пространства.

Вес отвечает за направление и важность признака:

- положительный вес увеличивает `z`, когда признак растет;
- отрицательный вес уменьшает `z`, когда признак растет;
- вес около нуля почти выключает влияние признака.

Смещение `b` сдвигает границу решения. Если `b` сильно отрицательный, нейрону нужно больше положительного сигнала от признаков, чтобы активироваться.

Один нейрон со ступенчатой активацией способен провести только одну линейную границу. Этого достаточно для простых случаев, но недостаточно для сложных данных, где классы переплетены. Поэтому нейроны объединяются в слои: несколько простых границ вместе могут образовывать более сложные области.

In [ ]:
def explain_neuron(inputs, weights, bias, activation):
    print("Подробный расчет нейрона")
    print("-" * 32)
    total = bias
    print(f"Начинаем со смещения bias = {bias:.3f}")
    for index, (x_i, w_i) in enumerate(zip(inputs, weights), start=1):
        contribution = x_i * w_i
        total += contribution
        print(f"x{index} * w{index} = {x_i:.3f} * {w_i:.3f} = {contribution:.3f}")
    print(f"z = {total:.3f}")
    print(f"activation(z) = {activation(total):.3f}")

explain_neuron(
    inputs=[0.65, 0.70, 0.20],
    weights=[3.0, 2.0, 0.5],
    bias=-2.5,
    activation=sigmoid,
)

## 4. Почему нужна функция активации

Активация делает модель нелинейной.

Если слой без активации записать как:

```text
y = W*x + b
```

и затем поставить еще один линейный слой:

```text
u = V*y + c
```

то получится:

```text
u = V*(W*x + b) + c
u = (V*W)*x + (V*b + c)
```

Это снова одно аффинное преобразование. Значит, много линейных слоев без активаций можно свернуть в один линейный слой.

Поэтому между слоями добавляют нелинейную функцию:

```text
y = sigma(W*x + b)
```

Теперь следующему слою приходит уже нелинейно преобразованный вектор, и вся сеть может описывать гораздо более сложные зависимости.

Интуитивно активация меняет характер модели. Линейная модель может только взвешивать признаки и складывать их. Нелинейная модель может реагировать на комбинации признаков: например, "если одновременно высокая влажность и высокая облачность", а не просто "чем больше влажность, тем больше ответ". Именно такие взаимодействия признаков часто делают нейронные сети полезными.

На первом этапе достаточно знать три активации:

| Активация | Идея | Диапазон |
|---|---|---|
| `step` | жесткое решение 0 или 1 | `{0, 1}` |
| `sigmoid` | мягкая вероятность | `(0, 1)` |
| `ReLU` | пропускает положительные значения | `[0, +inf)` |

В выходном слое бинарного классификатора удобно использовать `sigmoid`, потому что ее результат лежит между `0` и `1`. В скрытых слоях часто используют ReLU и похожие функции, потому что они просты, быстро вычисляются и хорошо работают в глубоких сетях.

In [ ]:
values = [-4, -2, -1, 0, 1, 2, 4]

print(f"{'z':>5} {'step':>7} {'sigmoid':>9} {'ReLU':>7}")
print("-" * 32)
for value in values:
    print(f"{value:>5.1f} {step(value):>7} {sigmoid(value):>9.4f} {relu(value):>7.1f}")

## 5. От нейрона к слою

Один нейрон возвращает одно число.

Слой из `k` нейронов возвращает вектор из `k` чисел:

```text
a = (a1, a2, ..., ak)
```

Если вход имеет размерность `n`, а в слое `k` нейронов, то у слоя есть:

```text
W - матрица размера k x n
b - вектор размера k
```

Формула слоя:

```text
z = W*x + b
a = sigma(z)
```

В развернутом виде для `j`-го нейрона:

```text
zj = wj1*x1 + wj2*x2 + ... + wjn*xn + bj
aj = sigma(zj)
```

Полносвязный слой называется полносвязным, потому что каждый входной признак соединен с каждым нейроном слоя. Если входов `n`, а нейронов `k`, то количество весов равно:

```text
n * k
```

И еще есть `k` смещений.

Содержательно слой можно понимать как набор детекторов. Каждый нейрон слоя реагирует на свой шаблон во входных данных. Один нейрон может быть чувствителен к высокой облачности, другой - к сочетанию ветра и влажности, третий - к признакам ясной погоды. В реальных сетях такие интерпретации не всегда очевидны, но математически идея остается такой: слой превращает исходные признаки в новое представление.

Это новое представление может быть удобнее для следующего слоя. Поэтому нейронная сеть не просто классифицирует исходные признаки, а постепенно строит промежуточные признаки, которые помогают решать задачу.

In [ ]:
def neuron_forward(inputs, weights, bias, activation):
    z = weighted_sum(inputs, weights, bias)
    return activation(z)

def dense_layer_forward(inputs, layer_weights, layer_biases, activation):
    outputs = []
    for weights, bias in zip(layer_weights, layer_biases):
        output = neuron_forward(inputs, weights, bias, activation)
        outputs.append(output)
    return outputs

hidden_weights = [
    [5.0, 4.0, 0.0],   # нейрон 1: облачно + влажно
    [0.0, 1.0, 5.0],   # нейрон 2: влажно + сильный ветер
    [-6.0, -1.0, 0.0], # нейрон 3: скорее ясная погода
]
hidden_biases = [-4.0, -3.0, 3.0]

sample = row_to_vector(dataset[3])
hidden_outputs = dense_layer_forward(sample, hidden_weights, hidden_biases, sigmoid)

print("Вход:", sample)
print("Выходы скрытого слоя:")
for i, value in enumerate(hidden_outputs, start=1):
    print(f"h{i} = {value:.4f}")

## 6. Полносвязный перцептрон как композиция функций

Многослойную полносвязную сеть можно записать как последовательность слоев.

Для сети с одним скрытым слоем:

```text
h = sigma(W1*x + b1)
y = phi(W2*h + b2)
```

Где:

- `x` - входной вектор;
- `W1`, `b1` - веса и смещения скрытого слоя;
- `h` - вектор скрытых признаков;
- `W2`, `b2` - веса и смещение выходного слоя;
- `phi` - активация выходного слоя.

Если подставить `h` во вторую формулу:

```text
y = phi(W2 * sigma(W1*x + b1) + b2)
```

Это уже не просто одна линейная формула, потому что внутри есть нелинейная `sigma`.

Для нашей задачи:

```text
[cloudiness, humidity, wind] -> [h1, h2, h3] -> umbrella_probability
```

Входной слой только хранит признаки. Скрытый слой строит промежуточное представление. Выходной слой превращает это представление в итоговую вероятность.

С точки зрения анализа функций такая сеть является композицией простых отображений. Первый слой переводит точку из исходного пространства признаков в пространство скрытых признаков. Второй слой берет уже это новое описание и строит ответ. Если слоев больше, то таких преобразований становится больше:

```text
x -> a1 -> a2 -> ... -> y
```

Именно композиция отличает глубокие модели от одной большой линейной формулы. Каждый слой может немного менять геометрию данных, делая задачу для следующего слоя проще.

## 7. Почему полносвязная сеть может приближать сложные функции

Полносвязные нейронные сети важны не только потому, что их удобно программировать. У них есть сильное математическое свойство: при достаточном числе нейронов они могут приближать очень широкий класс функций.

Одна из классических формулировок называется **теоремой универсальной аппроксимации**.

Неформально:

> Если функция `f` непрерывна на ограниченной замкнутой области, то полносвязная нейронная сеть с нелинейной активацией может приблизить `f` сколь угодно точно.

Более математически:

```text
Пусть f: K -> R непрерывна,
где K - компактное подмножество R^n.

Тогда для любого epsilon > 0 существует нейронная сеть F такая, что

|F(x) - f(x)| < epsilon

для всех x из K.
```

Здесь важно каждое слово. Область `K` должна быть компактной, то есть ограниченной и замкнутой. Функция `f` должна быть непрерывной. Число `epsilon` задает допустимую ошибку приближения. Если `epsilon` маленькое, требуется более точное приближение.

Интуитивно нейроны можно представить как простые строительные элементы функции. Один нейрон создает простую нелинейную форму. Несколько нейронов могут сложить более гибкую кривую или поверхность. Большое число нейронов позволяет приближать функции с большим количеством изгибов, переходов и локальных особенностей.

В одномерном случае это похоже на приближение сложной кривой набором простых кусочков. В многомерном случае вместо кривой появляется поверхность или гиперповерхность в пространстве признаков. Полносвязная сеть настраивает параметры так, чтобы эта поверхность проходила рядом с нужной зависимостью.

Что это означает:

- сеть может приблизить прямую, параболу, синусоиду и многие более сложные зависимости;
- точность приближения задается числом `epsilon`;
- чтобы сделать ошибку меньше, обычно требуется больше нейронов или более удобная архитектура;
- нелинейная активация принципиальна: без нее сеть не получает универсальной выразительности.

Что это **не** означает:

- теорема не говорит, что маленькая сеть справится с любой задачей;
- теорема не говорит, что обучение обязательно найдет нужные веса;
- теорема не гарантирует хорошую работу на данных вне области `K`;
- теорема не отменяет необходимость нормальных данных.

Поэтому теорема универсальной аппроксимации объясняет потенциальную выразительность полносвязных сетей, но не решает всю задачу машинного обучения. Она отвечает на вопрос "может ли такая функция существовать", но не отвечает полностью на вопрос "как быстро и надежно найти ее параметры по данным".

In [ ]:
network = {
    "hidden_weights": [
        [5.0, 4.0, 0.0],
        [0.0, 1.0, 5.0],
        [-6.0, -1.0, 0.0],
    ],
    "hidden_biases": [-4.0, -3.0, 3.0],
    "output_weights": [[4.0, 1.5, -2.0]],
    "output_biases": [-1.9],
}

def network_forward(inputs, network):
    hidden = dense_layer_forward(
        inputs,
        network["hidden_weights"],
        network["hidden_biases"],
        sigmoid,
    )
    output = dense_layer_forward(
        hidden,
        network["output_weights"],
        network["output_biases"],
        sigmoid,
    )
    return output[0], hidden

probability, hidden = network_forward(row_to_vector(dataset[3]), network)

print("Скрытый слой:", [round(v, 4) for v in hidden])
print("Вероятность, что зонт нужен:", round(probability, 4))

## 8. Forward pass по всему датасету

**Forward pass** - это вычисление значения функции `F(x; theta)` при уже заданных параметрах.

Для каждого объекта из датасета делаем:

```text
x -> h = sigma(W1*x + b1) -> y = sigmoid(W2*h + b2)
```

Получаем число `y` от `0` до `1`.

Для бинарной классификации это число превращается в класс через порог:

```text
prediction = 1, если y >= threshold
prediction = 0, если y < threshold
```

Порог `0.5` удобен, но не является законом. Если ложный пропуск опаснее ложной тревоги, порог можно уменьшить. Если ложная тревога слишком дорогая, порог можно увеличить.

Важный момент: forward pass не меняет сеть. Он только применяет уже существующую функцию к данным. Это похоже на подстановку значения в формулу. Если формула фиксирована, то для одного и того же входа она всегда вернет один и тот же выход.

При обучении ситуация другая: после вычисления ответа считается ошибка, а затем параметры корректируются. Но даже во время обучения прямой проход остается отдельной частью процесса: сначала сеть считает предсказание, потом уже это предсказание сравнивается с правильным ответом.

In [ ]:
def predict_class(probability, threshold=0.5):
    return 1 if probability >= threshold else 0

def bar(value, width=20):
    filled = round(value * width)
    return "#" * filled + "." * (width - filled)

threshold = 0.5

print(f"{'day':<4} {'target':>6} {'prob':>8} {'pred':>6}  bar")
print("-" * 52)

correct = 0
for row in dataset:
    probability, hidden = network_forward(row_to_vector(row), network)
    prediction = predict_class(probability, threshold)
    correct += int(prediction == row["umbrella"])
    print(
        f"{row['day']:<4} "
        f"{row['umbrella']:>6} "
        f"{probability:>8.3f} "
        f"{prediction:>6}  "
        f"{bar(probability)}"
    )

accuracy = correct / len(dataset)
print("-" * 52)
print(f"Accuracy на игрушечном датасете: {accuracy:.3f}")

## 9. Подробная трассировка одного примера

Рассмотрим один объект максимально подробно.

Возьмем день `G`:

```text
cloudiness = 0.30
humidity   = 0.85
wind       = 0.90
```

Человеческая интуиция: облачность не очень высокая, но влажность и ветер большие. Возможно, зонт пригодится.

Для сети это не рассуждение словами, а набор числовых операций. Сначала исходные признаки попадают в скрытый слой. Каждый скрытый нейрон считает свою взвешенную сумму и применяет активацию. Затем выходной нейрон получает уже не исходные признаки, а значения скрытых нейронов. Поэтому итоговое решение зависит не только от отдельных признаков, но и от того, какие промежуточные комбинации были построены скрытым слоем.

In [ ]:
def trace_network(row, network):
    inputs = row_to_vector(row)
    print(f"День {row['day']}")
    print(f"Входы: cloudiness={inputs[0]:.2f}, humidity={inputs[1]:.2f}, wind={inputs[2]:.2f}")
    print()

    hidden_values = []
    for neuron_index, (weights, bias) in enumerate(
        zip(network["hidden_weights"], network["hidden_biases"]),
        start=1,
    ):
        z = weighted_sum(inputs, weights, bias)
        a = sigmoid(z)
        hidden_values.append(a)
        print(f"Скрытый нейрон h{neuron_index}")
        print(f"  weights = {weights}, bias = {bias}")
        print(f"  z = {z:.4f}")
        print(f"  sigmoid(z) = {a:.4f}")
        print()

    output_weights = network["output_weights"][0]
    output_bias = network["output_biases"][0]
    z_output = weighted_sum(hidden_values, output_weights, output_bias)
    probability = sigmoid(z_output)
    prediction = predict_class(probability)

    print("Выходной нейрон")
    print(f"  hidden = {[round(v, 4) for v in hidden_values]}")
    print(f"  weights = {output_weights}, bias = {output_bias}")
    print(f"  z = {z_output:.4f}")
    print(f"  sigmoid(z) = {probability:.4f}")
    print(f"  prediction = {prediction}")
    print(f"  target = {row['umbrella']}")

trace_network(dataset[6], network)

## 10. Размерности и количество параметров

Архитектура:

```text
3 входа -> 3 скрытых нейрона -> 1 выход
```

Скрытый слой:

```text
W1 имеет размер 3 x 3
b1 имеет размер 3
```

Значит:

```text
параметров в W1: 3 * 3 = 9
параметров в b1: 3
итого: 12
```

Выходной слой:

```text
W2 имеет размер 1 x 3
b2 имеет размер 1
```

Значит:

```text
параметров в W2: 1 * 3 = 3
параметров в b2: 1
итого: 4
```

Всего:

```text
12 + 4 = 16 параметров
```

Общая формула для полносвязного слоя:

```text
если входов n, а нейронов k, то параметров n*k + k
```

Или:

```text
k * (n + 1)
```

Плюс один внутри скобок соответствует смещению каждого нейрона.

Количество параметров быстро растет. Если входов `1000`, а в слое `500` нейронов, то только весов будет `1000 * 500 = 500000`, и еще `500` смещений. С одной стороны, большое число параметров дает модели гибкость. С другой стороны, оно требует больше данных, вычислений и аккуратности при обучении.

Именно поэтому архитектура сети является важным выбором. Слишком маленькая сеть может не выразить нужную зависимость. Слишком большая сеть может оказаться избыточной, медленной и склонной подстраиваться под шум в данных.

In [ ]:
def count_layer_parameters(layer_weights, layer_biases):
    weight_count = 0
    for neuron_weights in layer_weights:
        weight_count += len(neuron_weights)
    bias_count = len(layer_biases)
    return weight_count + bias_count

hidden_parameter_count = count_layer_parameters(
    network["hidden_weights"],
    network["hidden_biases"],
)
output_parameter_count = count_layer_parameters(
    network["output_weights"],
    network["output_biases"],
)

print("Параметров в скрытом слое:", hidden_parameter_count)
print("Параметров в выходном слое:", output_parameter_count)
print("Всего параметров:", hidden_parameter_count + output_parameter_count)

## 11. Архитектура, forward pass и обучение

Важно разделять три разных вопроса.

**Архитектура:**

```text
Какая функция задана по форме?
Сколько слоев, сколько нейронов, какие активации?
```

**Forward pass:**

```text
Как посчитать F(x; theta), если параметры theta уже известны?
```

**Обучение:**

```text
Как подобрать theta, чтобы F(x; theta) давала хорошие ответы?
```

В этом ноутбуке разобраны архитектура и forward pass. Параметры заданы вручную, поэтому сеть не обучается, а только демонстрирует механизм вычисления.

Такое разделение полезно методически и математически. Архитектура определяет семейство функций, из которого мы выбираем модель. Forward pass показывает, как конкретная функция из этого семейства вычисляет ответ. Обучение выбирает конкретные параметры внутри заданного семейства.

Если говорить кратко:

```text
архитектура задает форму;
параметры задают конкретную функцию;
forward pass применяет эту функцию;
обучение меняет параметры.
```

После понимания прямого прохода становится естественным следующий шаг: ввести функцию потерь, измерить ошибку предсказаний и обсудить, как изменение весов влияет на эту ошибку. Именно к этому приводит backpropagation, но сам backpropagation имеет смысл только после того, как ясно, что сеть вычисляет в прямом направлении.